# 04 · Live error shapes

`errors.py` parses pxwebpy's error *string* (there is no `.response` attached) to
recover a status code and a detail message. Every test of that so far used
**synthetic** strings we wrote ourselves in `tests/test_errors.py`.

This notebook triggers **real** failures against the live API and checks the
assumption holds.

In [1]:
import warnings

from ssb_statistikkbanken import SSB, SsbError

warnings.simplefilter("ignore")
ssb = SSB()

## 1 · Pre-flight validation (never reaches SSB)

Variable and value names are checked against cached metadata *before* any
request, so typos come back as suggestions rather than HTTP errors.

In [2]:
attempts = [
    ("unknown variable", lambda: ssb.fetch("08329", Skadegrader="01")),
    ("unknown value code", lambda: ssb.fetch("08329", Skadegrad="99")),
    ("wildcard matching nothing", lambda: ssb.fetch("08329", Skadegrad="zzz*")),
]
for label, attempt in attempts:
    try:
        attempt()
        print(f"{label}: NO ERROR")
    except SsbError as exc:
        print(f"### {label} -> {type(exc).__name__}")
        print(f"{exc}\n")

### unknown variable -> UnknownVariableError
'Skadegrader' is not a variable in table 08329.
  Did you mean: Skadegrad (skadegrad)?
  Available: Alder2, Kjonn, Skadegrad, VegtrTrafGrp, Ulukkesgruppe, ContentsCode, Tid
  Tip: ssb.variables("08329")

### unknown value code -> UnknownValueError
'99' is not a value code for Skadegrad (skadegrad) in table 08329.
  Tip: ssb.values("08329", "Skadegrad")

### wildcard matching nothing -> UnknownValueError
Wildcard 'zzz*' matched no value codes for Skadegrad (skadegrad) in table 08329.
  Tip: ssb.values("08329", "Skadegrad")



## 2 · Real HTTP failures

Now errors that genuinely come back from SSB, so the string parsing in
`wrap_pxweb_errors` is doing real work.

In [3]:
try:
    ssb.describe("99999")
except SsbError as exc:
    print(f"type   : {type(exc).__name__}")
    print(f"status : {exc.status_code}")
    print(f"detail : {exc.detail!r}")

type   : TableNotFoundError
status : 404
detail : 'Non-existent table'


A 404 parses cleanly into `TableNotFoundError` with the status and SSB's own
detail text recovered from the message string. The assumption holds.

### An unknown code list → HTTP **500**

In [4]:
try:
    ssb.fetch("14346", code_list={"Region": "vs_DoesNotExist"}, latest=1)
except SsbError as exc:
    print(f"type   : {type(exc).__name__}")
    print(f"status : {exc.status_code}")
    print(f"detail : {exc.detail!r}")

type   : SsbHTTPError
status : 500
detail : None


SSB answers a **client** mistake (a typo'd code list id) with a **server** error
and sends **no `detail` body**, so there is nothing in the response worth
showing the user.

**Fixed (F-04):** since the response carries nothing actionable, the wrapper
supplies the context itself — it says plainly that this was a server error and,
when a `code_list=` was part of the request, names the ids to check. The parsing
was always correct; the information simply was not there to parse.

## 3 · The batching hole, demonstrated live

`top(n)`, `from(x)`, `range(x,y)` and `?` are resolved **by SSB**, so neither the
wrapper nor pxwebpy can count the cells they will produce. pxwebpy counts each as
a single cell, so no batching happens — and an oversized request goes out intact.

Notebook 03 fetched **898,560 cells successfully** using an explicit month list.
The same data requested with `top(330)` is far bigger and cannot be split:

In [5]:
try:
    ssb.fetch(
        "08329",
        Alder2="*",
        Skadegrad="*",
        VegtrTrafGrp="*",
        Ulukkesgruppe="*",
        Kjonn="*",
        time="top(330)",
    )
    print("unexpected success")
except SsbError as exc:
    print(f"type   : {type(exc).__name__}")
    print(f"status : {exc.status_code}")
    print(f"detail : {exc.detail!r}")

type   : BadSelectionError
status : 400
detail : 'Too many cells selected'


`400 Too many cells selected`, straight from SSB — the documented risk, now
confirmed end to end rather than merely predicted.

The mitigation already in the docs is the right one: prefer `latest=n`, an
explicit list, or `*` (all of which the wrapper can count and batch) over PxWeb
expressions when the selection is large. The wrapper does warn when it cannot
size a selection:

In [6]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    ssb.fetch("08329", Skadegrad="01", time="top(3)")
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")